In [1]:
import random
import numpy as np
from pathlib import Path
from the_well.data import WellDataset

import torch
from torch.utils.data import DataLoader

from modules import *
from losses import *
from datasets import TRMDataset

In [ ]:
SEED         = 42
EPOCHS       = 15
BATCH_SIZE   = 16
LR           = 1e-3

DATASET_PATH = "/mnt/storage_C1/BILL_pino/the_well/datasets/turbulent_radiative_layer_2D"
OUTPUT_DIR   = "./models"

MODES1       = 16   # Modos de Fourier na primeira dimensão espacial (x)
MODES2       = 16   # Modos de Fourier na segunda dimensão espacial (y)
WIDTH        = 32   # Número de canais internos (largura do modelo)
DEPTH        = 4    # Quantidade de camadas de Fourier
PROJ_DIM     = 128  # Dimensão da MLP de projeção para a saída

In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"Torch: {torch.__version__}  |  Torchvision: {__import__('torchvision').__version__}")

Dispositivo: cuda
Torch: 2.11.0+cu128  |  Torchvision: 0.26.0+cu128


In [ ]:
train_dataset = WellDataset(
    path=DATASET_PATH,
    well_split_name="train"
)

validation_dataset = WellDataset(
    path=DATASET_PATH,
    well_split_name="valid"
)

train_ds = TRMDataset(train_dataset)
val_ds = TRMDataset(validation_dataset)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=False,
    prefetch_factor=4,
)

### Teste 1

In [5]:
x, y = train_ds[0]
in_dim, out_dim = x.shape[-1], y.shape[-1] 

model = FNO2d(
    modes1=MODES1,
    modes2=MODES2,
    width=WIDTH,
    in_dim=in_dim,
    out_dim=out_dim,
    depth=DEPTH,
    proj_dim=PROJ_DIM
).cuda()
model = torch.compile(model)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
)

criterion = relative_l2_loss

In [6]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS,
    checkpoint_dir=OUTPUT_DIR
)

Epoch 1/15:   0%|          | 0/450 [00:00<?, ?it/s]/home/al.bruno.paula/simul/env/lib/python3.12/site-packages/torch/_inductor/lowering.py:2212: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
W0622 19:53:24.116000 18685 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode
Epoch 1/15: 100%|██████████| 450/450 [03:35<00:00,  2.09it/s, loss=0.158832]


Epoch 001 | train = 0.158832 | val = 0.112398 | best_val = 0.112398


Epoch 2/15: 100%|██████████| 450/450 [01:55<00:00,  3.90it/s, loss=0.104977]


Epoch 002 | train = 0.104977 | val = 0.107976 | best_val = 0.107976


Epoch 3/15: 100%|██████████| 450/450 [01:19<00:00,  5.64it/s, loss=0.101013]


...

Epoch 013 | train = 0.081778 | val = 0.098374 | best_val = 0.098374


Epoch 14/15: 100%|██████████| 450/450 [01:16<00:00,  5.89it/s, loss=0.081223]


Epoch 014 | train = 0.081223 | val = 0.098384 | best_val = 0.098374


Epoch 15/15: 100%|██████████| 450/450 [01:19<00:00,  5.63it/s, loss=0.080893]


Epoch 015 | train = 0.080893 | val = 0.098307 | best_val = 0.098307


Resultado Teste 1

| train = 0.080893 | val = 0.098307 | best_val = 0.098307

### Teste 2

In [ ]:
x, y = train_ds[0]
in_dim, out_dim = x.shape[-1], y.shape[-1] 

model = FNO2d_v1(
    modes1=MODES1,
    modes2=MODES2,
    width=WIDTH,
    width1= WIDTH,
    in_dim=in_dim,
    out_dim=out_dim,
    depth=DEPTH,
    proj_dim=PROJ_DIM
).cuda()
model = torch.compile(model)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
)

criterion = relative_l2_loss

In [8]:
history2 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS,
    checkpoint_dir=OUTPUT_DIR
)

Epoch 1/15: 100%|██████████| 450/450 [01:30<00:00,  4.99it/s, loss=0.298728]


Epoch 001 | train = 0.298728 | val = 0.108338 | best_val = 0.108338


Epoch 2/15: 100%|██████████| 450/450 [01:24<00:00,  5.32it/s, loss=0.100905]


Epoch 002 | train = 0.100905 | val = 0.102738 | best_val = 0.102738


Epoch 3/15: 100%|██████████| 450/450 [01:24<00:00,  5.35it/s, loss=0.094801]


...

Epoch 013 | train = 0.058918 | val = 0.075016 | best_val = 0.075016


Epoch 14/15: 100%|██████████| 450/450 [01:24<00:00,  5.32it/s, loss=0.058085]


Epoch 014 | train = 0.058085 | val = 0.074871 | best_val = 0.074871


Epoch 15/15: 100%|██████████| 450/450 [01:24<00:00,  5.34it/s, loss=0.057624]


Epoch 015 | train = 0.057624 | val = 0.074753 | best_val = 0.074753


Resultado teste 2

Epoch 015 | train = 0.057624 | val = 0.074753 | best_val = 0.074753

### Teste 3

In [9]:
x, y = train_ds[0]
in_dim, out_dim = x.shape[-1], y.shape[-1] 

model = FNO2d_v1(
    modes1=MODES1,
    modes2=MODES2,
    width=WIDTH,
    width1= WIDTH,
    in_dim=in_dim,
    out_dim=out_dim,
    depth=DEPTH+1,
    proj_dim=192
).cuda()
model = torch.compile(model)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
)

criterion = relative_l2_loss

In [10]:
history3 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS,
    checkpoint_dir=OUTPUT_DIR
)

Epoch 1/15: 100%|██████████| 450/450 [01:56<00:00,  3.85it/s, loss=0.253459]


Epoch 001 | train = 0.253459 | val = 0.107628 | best_val = 0.107628


Epoch 2/15: 100%|██████████| 450/450 [01:42<00:00,  4.39it/s, loss=0.099067]


Epoch 002 | train = 0.099067 | val = 0.102299 | best_val = 0.102299


Epoch 3/15: 100%|██████████| 450/450 [01:41<00:00,  4.42it/s, loss=0.091461]


...

Epoch 013 | train = 0.053388 | val = 0.072917 | best_val = 0.072917


Epoch 14/15: 100%|██████████| 450/450 [01:41<00:00,  4.45it/s, loss=0.052485]


Epoch 014 | train = 0.052485 | val = 0.072857 | best_val = 0.072857


Epoch 15/15: 100%|██████████| 450/450 [01:40<00:00,  4.46it/s, loss=0.051988]


Epoch 015 | train = 0.051988 | val = 0.072924 | best_val = 0.072857


Resultado teste 3

Epoch 015 | train = 0.051988 | val = 0.072924 | best_val = 0.072857